# Machine Learning-Based Wireless Intrusion Detection System (WIDS)

**Goal:** demonstrate detection of deauthentication-flood-like traffic using a Random Forest classifier.

> **Honesty and safety note:** This notebook uses a synthetic dataset only. It does not capture live Wi-Fi traffic, transmit attacks, disconnect clients, or block devices. Results measure performance on a held-out synthetic test split, not real-world Wi-Fi performance.


## 1. Install/import libraries

In [ ]:
# In Google Colab, these are commonly preinstalled. Uncomment if needed:
# %pip -q install numpy pandas matplotlib scikit-learn

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)

OUTPUT = Path("results")
OUTPUT.mkdir(exist_ok=True)


## 2. Generate synthetic wireless traffic

In [ ]:
rng = np.random.default_rng(42)
n_normal, n_attack = 2400, 800

normal = pd.DataFrame({
    "frame_type_code": rng.choice([0, 1, 2], n_normal, p=[0.22, 0.13, 0.65]),
    "signal_strength_dbm": np.clip(rng.normal(-58, 12, n_normal), -95, -25),
    "frames_per_second": np.clip(rng.gamma(2.2, 5.0, n_normal), 0, 45),
    "deauth_frames_per_second": rng.poisson(0.03, n_normal),
    "unique_destination_count": rng.integers(1, 5, n_normal),
    "label": 0
})
attack = pd.DataFrame({
    "frame_type_code": rng.choice([0, 1, 2], n_attack, p=[0.86, 0.08, 0.06]),
    "signal_strength_dbm": np.clip(rng.normal(-52, 10, n_attack), -90, -20),
    "frames_per_second": np.clip(rng.normal(115, 24, n_attack), 45, 220),
    "deauth_frames_per_second": np.clip(rng.poisson(42, n_attack), 12, 100),
    "unique_destination_count": np.clip(rng.poisson(18, n_attack), 5, 60),
    "label": 1
})
data = pd.concat([normal, attack], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
data.to_csv(OUTPUT / "synthetic_wireless_traffic.csv", index=False)
print("Dataset shape:", data.shape)
print("Class counts (0=normal, 1=malicious-like):")
display(data["label"].value_counts().rename(index={0:"Normal", 1:"Malicious-like"}))
display(data.head())


## 3. Train the Random Forest classifier

In [ ]:
FEATURES = [
    "frame_type_code",
    "signal_strength_dbm",
    "frames_per_second",
    "deauth_frames_per_second",
    "unique_destination_count"
]
X, y = data[FEATURES], data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=120, max_depth=8, class_weight="balanced", random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


## 4. Evaluate the model

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
report = classification_report(
    y_test, y_pred, labels=[0, 1],
    target_names=["Normal", "Malicious"], output_dict=True, zero_division=0
)

print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print("\nClassification report:")
print(classification_report(
    y_test, y_pred, labels=[0, 1],
    target_names=["Normal", "Malicious"], zero_division=0
))
print("Confusion matrix (rows=actual, columns=predicted; order Normal, Malicious):")
print(cm)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=["Normal", "Malicious"]).plot(
    ax=ax, cmap="Blues", colorbar=False
)
ax.set_title("WIDS Confusion Matrix — Synthetic Test Set")
plt.tight_layout()
plt.savefig(OUTPUT / "confusion_matrix.png", dpi=180)
plt.show()


## 5. Feature importance

In [ ]:
importance = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
ax = importance.plot(kind="barh", figsize=(7, 4.5), title="Random Forest Feature Importance")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.savefig(OUTPUT / "feature_importance.png", dpi=180)
plt.show()


## 6. Demonstrate alerts

In [ ]:
# Demonstrate alerts on a few synthetic attack-like rows.
demo = data[data["label"] == 1].head(10).copy()
demo["predicted_label"] = model.predict(demo[FEATURES])
for idx, row in demo.iterrows():
    if row["predicted_label"] == 1 or row["deauth_frames_per_second"] >= 25:
        print(
            f"ALERT: suspicious wireless activity | row={idx} | "
            f"frames/s={row['frames_per_second']:.1f} | "
            f"deauth/s={row['deauth_frames_per_second']} | "
            f"predicted={'MALICIOUS' if row['predicted_label'] else 'NORMAL'}"
        )
print("\nThis is an alert demonstration on synthetic rows, not a live wireless monitor.")


## 7. Save reproducible results

In [ ]:
metrics = {
    "dataset_type": "synthetic simulation",
    "rows": int(len(data)),
    "normal_rows": int((data.label == 0).sum()),
    "malicious_rows": int((data.label == 1).sum()),
    "features": FEATURES,
    "model": "RandomForestClassifier(n_estimators=120, max_depth=8)",
    "accuracy": float(accuracy),
    "confusion_matrix_labels": ["Normal", "Malicious"],
    "confusion_matrix": cm.tolist(),
    "classification_report": report,
    "note": "Synthetic-data results only; not real-world Wi-Fi performance."
}
(OUTPUT / "metrics.json").write_text(json.dumps(metrics, indent=2))
test_output = X_test.copy()
test_output["actual_label"] = y_test
test_output["predicted_label"] = y_pred
test_output.to_csv(OUTPUT / "test_predictions.csv", index=False)
print("Saved files:")
for p in sorted(OUTPUT.iterdir()):
    print("-", p)


## 8. What to explain in the demonstration
- **Feature extraction:** each row summarises a simulated wireless observation window.
- **Label:** `0` means normal; `1` means deauthentication-flood-like malicious traffic.
- **Train/test split:** 75% trains the model and 25% evaluates it on held-out synthetic examples.
- **Random Forest:** combines many decision trees and votes on the class.
- **Confusion matrix:** rows are actual labels; columns are predicted labels. False negatives are malicious examples incorrectly classified as normal.
- **Limitations:** synthetic data can be unrealistically separable. Real deployment needs authorised monitor-mode packet captures, robust lab labels, time-based testing, and false-positive evaluation.
